# 08x promotion vs non-promotion EDA

Descriptive EDA only. No modeling, no train/test split, no prediction, no SHAP, no Optuna, no segmentation, and no causal claim.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import math
import os
import shutil
import subprocess
import zipfile

import numpy as np
import pandas as pd

STEP = '08x_promotion_nonpromotion_EDA_260516'
START_TS = datetime.now()
warnings = []
errors = []

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / '.git').exists() and (p / 'park.ingyeom').exists():
            return p.resolve()
    raise RuntimeError('repo root not found from notebook runtime cwd')

ROOT = find_repo_root()
PARK = ROOT / 'park.ingyeom'
NB_DIR = PARK / 'notebook' / STEP
NB_PATH = NB_DIR / f'{STEP}.ipynb'
OUT_DIR = PARK / 'reports' / 'audits' / STEP
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP}_review_package.zip'
NOTE_PATH = PARK / 'note.md'
INPUT_06X = PARK / 'reports' / 'audits' / '06x_dataset_generation_260515'
INPUT_07X = PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515'
DATA_DIR = PARK / 'data'

for d in [NB_DIR, OUT_DIR, ZIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

existing_payload = [p for p in OUT_DIR.iterdir() if p.name != '.ipynb_checkpoints' and not p.name.startswith('run_')]
if existing_payload:
    archive_dir = OUT_DIR / ('run_' + START_TS.strftime('%Y%m%d_%H%M%S'))
    archive_dir.mkdir(parents=True, exist_ok=True)
    for p in existing_payload:
        shutil.move(str(p), str(archive_dir / p.name))

def inside_park(path):
    try:
        Path(path).resolve().relative_to(PARK.resolve())
        return True
    except Exception:
        return False

def write_csv(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def norm_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in {'true', 'yes', '1', 'y'}

print('ROOT =', ROOT)
print('PARK =', PARK)
print('OUT_DIR =', OUT_DIR)

ROOT = C:\Code\ott-churn-prediction
PARK = C:\Code\ott-churn-prediction\park.ingyeom
OUT_DIR = C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\08x_promotion_nonpromotion_EDA_260516


In [2]:
required_06x = [
    '06x_conservative_dataset.csv',
    '06x_expanded_dataset.csv',
    '06x_model_feature_lists.csv',
    '06x_dataset_schema_conservative.csv',
    '06x_dataset_schema_expanded.csv',
    '06x_scope_feature_policy.csv',
    '06x_caveat_register.csv',
    '06x_final_checks.csv',
]
required_07x = [
    '07x_feature_mapping_master.csv',
    '07x_AARRR_summary_by_feature_set.csv',
    '07x_conservative_AARRR_mapping.csv',
    '07x_expanded_AARRR_mapping.csv',
    '07x_scope_policy_handoff.csv',
    '07x_caveat_handoff.csv',
    '07x_downstream_EDA_handoff.csv',
    '07x_final_checks.csv',
]
raw_source_names = [
    '(광일)Membership_v2_with_derived_features.csv',
    'Membership_v2.csv',
    'View_History_v2.csv',
    'User_Mapping_v2.csv',
    'Movie_Master_v2.csv',
    'Membership_train.csv',
    '변수_합집합_비교_v3.csv',
]

def file_fingerprint(path):
    path = Path(path)
    if not path.exists():
        return {'sha256': '', 'mtime': '', 'size': '', 'status': 'missing'}
    h = hashlib.sha256()
    try:
        with path.open('rb') as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b''):
                h.update(chunk)
        stat = path.stat()
        return {
            'sha256': h.hexdigest(),
            'mtime': datetime.fromtimestamp(stat.st_mtime).isoformat(timespec='seconds'),
            'size': stat.st_size,
            'status': 'ok',
        }
    except Exception as exc:
        return {'sha256': '', 'mtime': '', 'size': '', 'status': f'error: {exc}'}

source_before = {name: file_fingerprint(DATA_DIR / name) for name in raw_source_names}
preflight_rows = []

def add_preflight(check, status, detail=''):
    preflight_rows.append({'check': check, 'status': status, 'detail': detail})

add_preflight('06x folder exists', 'PASS' if INPUT_06X.exists() else 'FAIL', str(INPUT_06X))
add_preflight('07x folder exists', 'PASS' if INPUT_07X.exists() else 'FAIL', str(INPUT_07X))
missing_06x = [name for name in required_06x if not (INPUT_06X / name).exists()]
missing_07x = [name for name in required_07x if not (INPUT_07X / name).exists()]
add_preflight('required 06x files exist', 'PASS' if not missing_06x else 'FAIL', '; '.join(missing_06x))
add_preflight('required 07x files exist', 'PASS' if not missing_07x else 'FAIL', '; '.join(missing_07x))
add_preflight('source fingerprint before captured', 'PASS', f'count={len(source_before)}')
add_preflight('output folder created', 'PASS' if OUT_DIR.exists() else 'FAIL', str(OUT_DIR))

In [3]:
def final_checks_pass(path):
    df = pd.read_csv(path)
    if 'status' not in df.columns:
        return False, 'status column missing'
    bad = df[~df['status'].astype(str).str.upper().eq('PASS')]
    if len(bad):
        return False, '; '.join(bad.iloc[:5].astype(str).agg(' | '.join, axis=1).tolist())
    return True, f'rows={len(df)}'

ok_06x, detail_06x = final_checks_pass(INPUT_06X / '06x_final_checks.csv')
ok_07x, detail_07x = final_checks_pass(INPUT_07X / '07x_final_checks.csv')
add_preflight('06x final checks pass', 'PASS' if ok_06x else 'FAIL', detail_06x)
add_preflight('07x final checks pass', 'PASS' if ok_07x else 'FAIL', detail_07x)

conservative = pd.read_csv(INPUT_06X / '06x_conservative_dataset.csv')
expanded = pd.read_csv(INPUT_06X / '06x_expanded_dataset.csv')
model_feature_lists = pd.read_csv(INPUT_06X / '06x_model_feature_lists.csv')
schema_cons = pd.read_csv(INPUT_06X / '06x_dataset_schema_conservative.csv')
schema_exp = pd.read_csv(INPUT_06X / '06x_dataset_schema_expanded.csv')
scope_policy = pd.read_csv(INPUT_06X / '06x_scope_feature_policy.csv')
caveat_register = pd.read_csv(INPUT_06X / '06x_caveat_register.csv')
mapping = pd.read_csv(INPUT_07X / '07x_feature_mapping_master.csv')
arr_summary = pd.read_csv(INPUT_07X / '07x_AARRR_summary_by_feature_set.csv')
map_cons = pd.read_csv(INPUT_07X / '07x_conservative_AARRR_mapping.csv')
map_exp = pd.read_csv(INPUT_07X / '07x_expanded_AARRR_mapping.csv')
scope_handoff = pd.read_csv(INPUT_07X / '07x_scope_policy_handoff.csv')
caveat_handoff = pd.read_csv(INPUT_07X / '07x_caveat_handoff.csv')
downstream_07x = pd.read_csv(INPUT_07X / '07x_downstream_EDA_handoff.csv')

add_preflight('conservative dataset loaded', 'PASS', str(conservative.shape))
add_preflight('expanded dataset loaded', 'PASS', str(expanded.shape))
add_preflight('07x mapping loaded', 'PASS', str(mapping.shape))
add_preflight('07x downstream EDA handoff loaded', 'PASS', str(downstream_07x.shape))

required_base_cols = {'USER_KEY', 'is_repurchase'}
split_checks = []
promotion_split_available = False
conservative_split = None
expanded_split = None

if 'is_promotion' in expanded.columns:
    expanded_split = expanded['is_promotion']
    split_checks.append('expanded has is_promotion')
else:
    split_checks.append('expanded lacks is_promotion')

if 'is_promotion' in conservative.columns:
    conservative_split = conservative['is_promotion']
    promotion_split_available = True
    split_checks.append('conservative has direct is_promotion')
elif expanded_split is not None and len(conservative) == len(expanded) and required_base_cols.issubset(conservative.columns) and required_base_cols.issubset(expanded.columns):
    rowwise_match = conservative[['USER_KEY', 'is_repurchase']].reset_index(drop=True).equals(expanded[['USER_KEY', 'is_repurchase']].reset_index(drop=True))
    if rowwise_match:
        conservative_split = expanded_split.reset_index(drop=True)
        promotion_split_available = True
        split_checks.append('conservative split uses row-aligned expanded is_promotion after USER_KEY/is_repurchase validation')
    else:
        split_checks.append('row-aligned USER_KEY/is_repurchase validation failed')

if expanded_split is not None and conservative_split is not None:
    promotion_split_available = promotion_split_available and set(pd.Series(expanded_split).dropna().unique()).issubset({0, 1}) and set(pd.Series(conservative_split).dropna().unique()).issubset({0, 1})

add_preflight('promotion split available', 'PASS' if promotion_split_available else 'FAIL', '; '.join(split_checks))
stop_reason = 'ready' if all(r['status'] == 'PASS' for r in preflight_rows) else 'preflight_failed_review_required'
add_preflight('stop_reason', 'PASS' if stop_reason == 'ready' else 'FAIL', stop_reason)
write_csv(pd.DataFrame(preflight_rows), '08x_preflight_input_validation.csv')

if stop_reason != 'ready':
    raise RuntimeError(stop_reason)

In [4]:
datasets = {
    'conservative_safe_22': {'df': conservative, 'split': pd.Series(conservative_split).reset_index(drop=True)},
    'expanded_feature_set': {'df': expanded, 'split': pd.Series(expanded_split).reset_index(drop=True)},
}

mapping['use_as_feature_norm'] = mapping['use_as_feature'].map(norm_bool)
mapping['caveat_flag_norm'] = mapping.get('caveat_flag', False).map(norm_bool) if 'caveat_flag' in mapping.columns else False

def feature_rows(feature_set_name):
    rows = mapping[(mapping['feature_set_name'] == feature_set_name) & (mapping['use_as_feature_norm'])].copy()
    rows = rows[~rows['safe_model_feature_name'].isin(['USER_KEY', 'is_repurchase'])]
    return rows

def target_counts(df, split, group_value=None):
    work = df if group_value is None else df[pd.Series(split).values == group_value]
    rep1 = int((work['is_repurchase'] == 1).sum())
    rep0 = int((work['is_repurchase'] == 0).sum())
    rate = rep1 / len(work) if len(work) else np.nan
    return len(work), rep0, rep1, rate

scope_rows = []
for fs, payload in datasets.items():
    df = payload['df']
    split = payload['split']
    fcount = int(len(feature_rows(fs)))
    scope_prefix = 'conservative' if fs == 'conservative_safe_22' else 'expanded'
    for suffix, group_value in [('overall', None), ('promotion_only', 1), ('nonpromotion_only', 0)]:
        rows, rep0, rep1, rate = target_counts(df, split, group_value)
        scope_rows.append({
            'feature_set_name': fs,
            'scope_name': f'{scope_prefix}_{suffix}',
            'row_count': rows,
            'column_count': int(df.shape[1]),
            'feature_count': fcount,
            'promotion_0_count': int((split == 0).sum()) if group_value is None else (rows if group_value == 0 else 0),
            'promotion_1_count': int((split == 1).sum()) if group_value is None else (rows if group_value == 1 else 0),
            'repurchase_0_count': rep0,
            'repurchase_1_count': rep1,
            'repurchase_rate': rate,
            'notes': 'conservative split key is row-aligned from expanded after validation' if fs == 'conservative_safe_22' else 'expanded contains is_promotion split key',
        })
dataset_scope_summary = pd.DataFrame(scope_rows)
write_csv(dataset_scope_summary, '08x_dataset_scope_summary.csv')

target_rows = []
for fs, payload in datasets.items():
    df = payload['df']
    split = payload['split']
    rates = {}
    counts = {}
    for label, value in [('promotion', 1), ('nonpromotion', 0)]:
        rows, rep0, rep1, rate = target_counts(df, split, value)
        rates[label] = rate
        counts[label] = (rows, rep0, rep1)
    for label, other in [('promotion', 'nonpromotion'), ('nonpromotion', 'promotion')]:
        rows, rep0, rep1 = counts[label]
        rate = rates[label]
        direction = 'higher' if rate > rates[other] else ('lower' if rate < rates[other] else 'similar')
        target_rows.append({
            'feature_set_name': fs,
            'group_name': label,
            'row_count': rows,
            'repurchase_1_count': rep1,
            'repurchase_0_count': rep0,
            'repurchase_rate': rate,
            'churn_proxy_rate': 1 - rate if pd.notna(rate) else np.nan,
            'rate_difference_vs_other_group': rate - rates[other] if pd.notna(rate) and pd.notna(rates[other]) else np.nan,
            'safe_interpretation': f'{label} group shows {direction} observed repurchase rate than {other} group.',
            'forbidden_interpretation': 'promotion caused churn/repurchase difference.',
        })
target_summary = pd.DataFrame(target_rows)
write_csv(target_summary, '08x_promotion_target_summary.csv')

WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/08x_promotion_nonpromotion_EDA_260516/08x_promotion_target_summary.csv')

In [5]:
def is_binary_series(s):
    vals = pd.Series(s).dropna().unique()
    return len(vals) <= 2 and set(vals).issubset({0, 1, 0.0, 1.0})

def smd_from_values(a, b):
    a = pd.to_numeric(a, errors='coerce').dropna()
    b = pd.to_numeric(b, errors='coerce').dropna()
    if len(a) == 0 or len(b) == 0:
        return np.nan
    pooled = math.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2) if len(a) > 1 and len(b) > 1 else 0
    if pooled == 0 or pd.isna(pooled):
        return 0.0 if a.mean() == b.mean() else np.nan
    return (a.mean() - b.mean()) / pooled

numeric_rows = []
binary_rows = []
for fs, payload in datasets.items():
    df = payload['df']
    split = payload['split']
    features = feature_rows(fs)
    for _, meta in features.iterrows():
        feat = meta['safe_model_feature_name']
        if feat in ['USER_KEY', 'is_repurchase', 'is_promotion'] or feat not in df.columns:
            continue
        s = pd.to_numeric(df[feat], errors='coerce')
        promo = s[split.values == 1]
        nonpromo = s[split.values == 0]
        miss = int(df[feat].isna().sum())
        caveat_flag = bool(norm_bool(meta.get('caveat_flag', False)))
        caveat_reason = '' if pd.isna(meta.get('caveat_reason', '')) else str(meta.get('caveat_reason', ''))
        needs_review = bool(norm_bool(meta.get('needs_user_review', False)))
        smd = smd_from_values(promo, nonpromo)
        base = {
            'feature_set_name': fs,
            'safe_model_feature_name': feat,
            'AARRR_stage': meta.get('AARRR_stage', ''),
            'feature_family': meta.get('feature_family', ''),
            'timing_family': meta.get('timing_family', ''),
            'promotion_n': int(promo.notna().sum()),
            'nonpromotion_n': int(nonpromo.notna().sum()),
            'standardized_mean_difference': smd,
            'missing_count': miss,
            'caveat_flag': caveat_flag,
            'caveat_reason': caveat_reason,
            'needs_user_review': needs_review,
            'interpretation_guardrail': 'Observed promotion vs non-promotion difference only; not causal, not feature selection, not model evidence.',
        }
        if is_binary_series(s):
            pr = promo.mean()
            nr = nonpromo.mean()
            binary_rows.append({
                **base,
                'promotion_rate': pr,
                'nonpromotion_rate': nr,
                'rate_diff_promotion_minus_nonpromotion': pr - nr,
                'relative_rate_ratio': pr / nr if nr not in [0, np.nan] and pd.notna(nr) else np.nan,
            })
        else:
            pm = promo.mean()
            nm = nonpromo.mean()
            pmed = promo.median()
            nmed = nonpromo.median()
            numeric_rows.append({
                **base,
                'promotion_mean': pm,
                'nonpromotion_mean': nm,
                'mean_diff_promotion_minus_nonpromotion': pm - nm,
                'promotion_median': pmed,
                'nonpromotion_median': nmed,
                'median_diff_promotion_minus_nonpromotion': pmed - nmed,
                'promotion_std': promo.std(ddof=1),
                'nonpromotion_std': nonpromo.std(ddof=1),
                'effect_size_abs': abs(smd) if pd.notna(smd) else np.nan,
            })

numeric_comp = pd.DataFrame(numeric_rows)
binary_comp = pd.DataFrame(binary_rows)
write_csv(numeric_comp, '08x_numeric_feature_group_comparison.csv')
write_csv(binary_comp, '08x_binary_feature_group_comparison.csv')
print('numeric rows', len(numeric_comp), 'binary rows', len(binary_comp))

numeric rows 69 binary rows 32


In [6]:
metric_frames = []
if len(numeric_comp):
    metric_frames.append(numeric_comp.assign(metric_type='numeric', effect_size_or_abs_smd=numeric_comp['effect_size_abs'], promotion_value=numeric_comp['promotion_mean'], nonpromotion_value=numeric_comp['nonpromotion_mean'], difference=numeric_comp['mean_diff_promotion_minus_nonpromotion']))
if len(binary_comp):
    metric_frames.append(binary_comp.assign(metric_type='binary', effect_size_or_abs_smd=binary_comp['standardized_mean_difference'].abs(), promotion_value=binary_comp['promotion_rate'], nonpromotion_value=binary_comp['nonpromotion_rate'], difference=binary_comp['rate_diff_promotion_minus_nonpromotion']))
all_metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()

family_rows = []
stage_rows = []
for fs in datasets:
    fs_features = feature_rows(fs)
    fs_metrics = all_metrics[all_metrics['feature_set_name'] == fs].copy() if len(all_metrics) else pd.DataFrame()
    for (stage, family), group in fs_features.groupby(['AARRR_stage', 'feature_family'], dropna=False):
        gm = fs_metrics[(fs_metrics['AARRR_stage'] == stage) & (fs_metrics['feature_family'] == family)]
        top = gm.sort_values('effect_size_or_abs_smd', ascending=False)['safe_model_feature_name'].head(5).tolist() if len(gm) else []
        family_rows.append({
            'feature_set_name': fs,
            'AARRR_stage': stage,
            'feature_family': family,
            'feature_count': int(len(group)),
            'compared_feature_count': int(len(gm)),
            'avg_abs_standardized_difference': gm['effect_size_or_abs_smd'].mean() if len(gm) else np.nan,
            'max_abs_standardized_difference': gm['effect_size_or_abs_smd'].max() if len(gm) else np.nan,
            'top_diff_features': ', '.join(top),
            'caveat_count': int(gm['caveat_flag'].sum()) if len(gm) else 0,
            'needs_user_review_count': int(gm['needs_user_review'].sum()) if len(gm) else 0,
            'summary_note': 'Descriptive family summary only; not removal, importance, or approval evidence.',
        })
    for stage in sorted(set(fs_features['AARRR_stage'].dropna().tolist() + ['Referral'])):
        group = fs_features[fs_features['AARRR_stage'] == stage]
        gm = fs_metrics[fs_metrics['AARRR_stage'] == stage]
        top = gm.sort_values('effect_size_or_abs_smd', ascending=False)['safe_model_feature_name'].head(5).tolist() if len(gm) else []
        note = 'Referral has no directly observed feature in this dataset; do not claim Referral was validated.' if stage == 'Referral' and len(group) == 0 else 'Observed difference summary only.'
        stage_rows.append({
            'feature_set_name': fs,
            'AARRR_stage': stage,
            'feature_count': int(len(group)),
            'compared_feature_count': int(len(gm)),
            'avg_abs_standardized_difference': gm['effect_size_or_abs_smd'].mean() if len(gm) else np.nan,
            'max_abs_standardized_difference': gm['effect_size_or_abs_smd'].max() if len(gm) else np.nan,
            'top_diff_features': ', '.join(top),
            'interpretation_guardrail': 'AARRR stage summary is descriptive only and cannot establish promotion effect.',
            'notes': note,
        })

feature_family_summary = pd.DataFrame(family_rows)
arr_stage_summary = pd.DataFrame(stage_rows)
write_csv(feature_family_summary, '08x_feature_family_summary.csv')
write_csv(arr_stage_summary, '08x_AARRR_stage_summary.csv')

top_rows = []
for fs in datasets:
    topm = all_metrics[all_metrics['feature_set_name'] == fs].sort_values('effect_size_or_abs_smd', ascending=False).head(20).reset_index(drop=True)
    for idx, row in topm.iterrows():
        top_rows.append({
            'feature_set_name': fs,
            'rank': idx + 1,
            'safe_model_feature_name': row['safe_model_feature_name'],
            'AARRR_stage': row['AARRR_stage'],
            'feature_family': row['feature_family'],
            'timing_family': row['timing_family'],
            'metric_type': row['metric_type'],
            'effect_size_or_abs_smd': row['effect_size_or_abs_smd'],
            'promotion_value': row['promotion_value'],
            'nonpromotion_value': row['nonpromotion_value'],
            'difference': row['difference'],
            'caveat_flag': row['caveat_flag'],
            'caveat_reason': row['caveat_reason'],
            'needs_user_review': row['needs_user_review'],
            'recommended_next_step': 'Review descriptively in 09x/10x, then audit redundancy/leakage in 10x or 11x before any modeling use.',
            'not_a_feature_selection_decision': True,
        })
top_observed = pd.DataFrame(top_rows)
write_csv(top_observed, '08x_top_observed_differences_for_review.csv')

WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/08x_promotion_nonpromotion_EDA_260516/08x_top_observed_differences_for_review.csv')

In [7]:
caveat_rows = [
    ('C01', 'observed difference only', 'all', 'all', 'promotion and non-promotion groups show observed differences.', 'promotion caused the observed difference.', 'Use observed difference language only.'),
    ('C02', 'no causal claim', 'all', 'all', 'This EDA does not estimate causal effects.', '100-won deal caused churn or repurchase.', 'Never use causal wording.'),
    ('C03', 'no uplift claim', 'all', 'all', 'No uplift was estimated.', 'promotion created uplift.', 'Do not use uplift wording.'),
    ('C04', 'no marketing effectiveness claim', 'all', 'all', 'Marketing effectiveness was not tested.', 'promotion was effective or ineffective.', 'Keep to descriptive group differences.'),
    ('C05', 'SHAP not performed', 'all', 'all', 'SHAP was not performed in 08x.', 'SHAP shows feature importance.', 'Leave SHAP to a later approved stage.'),
    ('C06', 'modeling not performed', 'all', 'all', 'No model was fit.', 'model performance improved.', 'No model evidence exists in 08x.'),
    ('C07', 'segmentation not performed', 'all', 'all', 'No final segment was created.', 'this is the final segment.', 'Do not create segment claims.'),
    ('C08', 'is_promotion split key caveat', 'is_promotion', 'Acquisition', 'is_promotion is a split key for this EDA.', 'is_promotion is an outcome or causal treatment estimate.', 'Use only as group split key.'),
    ('C09', 'is_churn_prevented historical context caveat', 'is_churn_prevented', 'Retention', 'is_churn_prevented requires historical context review.', 'is_churn_prevented proves prevention.', 'Treat as descriptive and caveated.'),
    ('C10', 'cold_start_fixed caveat', 'is_cold_start_3d_fixed; is_cold_start_7d_fixed', 'Activation', 'Fixed cold-start fields are row-level hotfix outputs.', 'original cold_start fields were used.', 'Name fixed fields explicitly.'),
    ('C11', 'old_movie_ratio_5y 9-row mismatch caveat', 'old_movie_ratio_5y', 'Retention', 'Known 9-row mismatch caveat remains.', 'old_movie_ratio_5y is fully resolved.', 'Carry caveat forward.'),
    ('C12', 'genre multi-category caveat', 'genre ratio fields', 'Retention', 'Genre ratios can reflect multi-category movie records.', 'genre fields are mutually exclusive categories.', 'Keep multi-category caveat.'),
    ('C13', 'under_1m/5m <= threshold caveat', 'watch_ratio_under_1m; watch_ratio_under_5m', 'Retention', 'Threshold features depend on <= cutoff convention.', 'threshold interpretation is self-evident without definition.', 'State threshold caveat when interpreting.'),
    ('C14', 'Optuna not performed', 'all', 'all', 'Optuna tuning was not performed.', 'Optuna selected a feature set.', 'No tuning language in 08x.'),
]
caveat_guardrail = pd.DataFrame(caveat_rows, columns=['caveat_id', 'caveat_topic', 'applies_to_feature', 'applies_to_stage', 'safe_claim', 'unsafe_claim', 'required_wording'])
write_csv(caveat_guardrail, '08x_caveat_and_claim_guardrail.csv')

handoff_rows = []
for fs in datasets:
    fs_top = top_observed[top_observed['feature_set_name'] == fs]['safe_model_feature_name'].head(12).tolist()
    fs_families = feature_family_summary[feature_family_summary['feature_set_name'] == fs].sort_values('max_abs_standardized_difference', ascending=False)['feature_family'].dropna().head(8).tolist()
    handoff_rows.extend([
        {'downstream_step': '09x_promotion_x_repurchase_2x2_EDA', 'handoff_topic': 'feature candidates for 2x2 descriptive review', 'feature_set_name': fs, 'related_features': ', '.join(fs_top), 'reason': 'Largest observed promotion/non-promotion differences in 08x.', 'required_action': 'Compare by promotion x repurchase groups without causal wording.', 'risk_if_ignored': '09x may miss strong descriptive contrast candidates.', 'user_approval_required': False, 'notes': 'Candidates are not selected features.'},
        {'downstream_step': '10x_feature_EDA', 'handoff_topic': 'feature families for distribution EDA', 'feature_set_name': fs, 'related_features': ', '.join(fs_families), 'reason': 'Families summarize observed differences.', 'required_action': 'Inspect distributions and missingness without removing features.', 'risk_if_ignored': 'Family-level distribution issues may be missed.', 'user_approval_required': False, 'notes': 'Feature family summary is not an importance ranking.'},
        {'downstream_step': '10x_or_11x', 'handoff_topic': 'multicollinearity and feature redundancy audit required', 'feature_set_name': fs, 'related_features': 'all numeric and binary use_as_feature fields', 'reason': '08x intentionally did not run redundancy audit.', 'required_action': 'Run VIF, pairwise correlation, redundancy cluster, near-constant, duplicate-like, and leakage-suspect audits.', 'risk_if_ignored': 'Expanded feature usage may be unstable or redundant.', 'user_approval_required': True, 'notes': 'No feature removal allowed without user approval.'},
        {'downstream_step': '11x_modeling_preflight', 'handoff_topic': 'conservative vs expanded feature usage confirmation', 'feature_set_name': fs, 'related_features': 'feature_set contract', 'reason': '08x kept conservative and expanded analyses separate.', 'required_action': 'Confirm actual feature list and scope before any model fit.', 'risk_if_ignored': 'Feature-set mixing or accidental is_promotion usage may occur.', 'user_approval_required': True, 'notes': '11x/12x must re-check actual expanded feature usage.'},
    ])
handoff_rows.append({'downstream_step': '11x/12x', 'handoff_topic': 'actual expanded feature count and usage re-check', 'feature_set_name': 'expanded_feature_set', 'related_features': 'expanded use_as_feature fields', 'reason': 'User requested actual expanded feature usage re-review.', 'required_action': 'Recount and validate expanded feature list before modeling.', 'risk_if_ignored': 'Expanded 80-feature assumption may drift from actual data contract.', 'user_approval_required': True, 'notes': 'Do not treat 08x as modeling pre-approval.'})
downstream_handoff = pd.DataFrame(handoff_rows)
write_csv(downstream_handoff, '08x_downstream_handoff.csv')

audit_items = [
    ('VIF', 'Detect multicollinearity among numeric features.', '10x or 11x', 'both', 'all numeric families', 'variance inflation factor', 'VIF table by feature'),
    ('pairwise correlation', 'Find highly correlated feature pairs.', '10x or 11x', 'both', 'all numeric families', 'Pearson/Spearman correlation matrix and high-pair table', 'high-correlation pair table'),
    ('feature family redundancy cluster', 'Summarize overlapping signals within feature families.', '10x or 11x', 'both', 'usage, retention, genre, device, registration', 'family-level clustering from correlations and definitions', 'redundancy cluster table'),
    ('near-constant feature', 'Identify features with little variation.', '10x or 11x', 'both', 'binary and numeric features', 'unique count and dominant-value share audit', 'near-constant audit table'),
    ('duplicate-like feature', 'Find columns carrying nearly identical information.', '10x or 11x', 'both', 'all features', 'exact and near-exact equality/correlation checks', 'duplicate-like feature table'),
    ('target leakage suspect', 'Review variables that may encode target or post-window information.', '11x', 'both', 'all features with policy caveats', 'policy review plus distribution checks by target', 'leakage suspect register'),
    ('SHAP interpretation grouping risk', 'Prevent redundant features from fragmenting later SHAP interpretation.', '11x or SHAP preflight', 'both', 'redundant families', 'grouping plan before SHAP', 'SHAP grouping risk handoff'),
]
redundancy_handoff = pd.DataFrame([
    {'audit_item': item, 'why_needed': why, 'suggested_step': step, 'target_feature_set': target, 'target_feature_family': fam, 'method': method, 'output_expected': out, 'removal_allowed': False, 'user_approval_required': True, 'notes': '08x does not perform this audit; handoff only.'}
    for item, why, step, target, fam, method, out in audit_items
])
write_csv(redundancy_handoff, '08x_redundancy_audit_handoff.csv')

WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/08x_promotion_nonpromotion_EDA_260516/08x_redundancy_audit_handoff.csv')

In [8]:
source_after = {name: file_fingerprint(DATA_DIR / name) for name in raw_source_names}
fp_rows = []
for name in raw_source_names:
    b = source_before[name]
    a = source_after[name]
    if b['status'] == 'missing':
        status = 'missing_before'
    elif a['status'] == 'missing':
        status = 'missing_after'
    elif b['status'].startswith('error') or a['status'].startswith('error'):
        status = 'error'
    elif b['sha256'] == a['sha256'] and b['mtime'] == a['mtime'] and b['size'] == a['size']:
        status = 'unchanged'
    else:
        status = 'changed'
    fp_rows.append({
        'file_path': str(DATA_DIR / name),
        'file_role': 'raw_source_fingerprint; Membership_train raw only, no missingness cause analysis' if name == 'Membership_train.csv' else 'raw_source_fingerprint',
        'sha256_before': b['sha256'],
        'sha256_after': a['sha256'],
        'mtime_before': b['mtime'],
        'mtime_after': a['mtime'],
        'size_before': b['size'],
        'size_after': a['size'],
        'status': status,
    })
fingerprint_df = pd.DataFrame(fp_rows)
write_csv(fingerprint_df, '08x_source_fingerprint_before_after.csv')

target_line = target_summary[['feature_set_name', 'group_name', 'row_count', 'repurchase_rate']].to_string(index=False)
top_line = top_observed.head(8)[['feature_set_name', 'rank', 'safe_model_feature_name', 'effect_size_or_abs_smd']].to_string(index=False)
readme = f'''# 08x promotion vs non-promotion EDA

## Purpose
08x performs descriptive promotion vs non-promotion EDA using 06x conservative and expanded datasets, with 07x feature mapping, AARRR mapping, and downstream EDA handoff as the interpretation frame.

## What This Step Does
- Compares observed promotion and non-promotion distributions.
- Keeps conservative_safe_22 and expanded_feature_set separate.
- Summarizes observed differences by feature family and AARRR stage.
- Creates 09x, 10x, and 11x handoff files.
- Verifies raw source CSV immutability with sha256, mtime, and size.

## What This Step Does Not Do
- No modeling, train/test split, prediction, SHAP, Optuna, segmentation, final segment, or final business recommendation.
- No new derived variables are persisted.
- No feature removal or final feature-use decision is made.
- No causal, uplift, or marketing effectiveness claim is made.

## Inputs
- 06x dataset generation: `{INPUT_06X}`
- 07x feature mapping and AARRR handoff: `{INPUT_07X}`
- Raw source fingerprint target folder: `{DATA_DIR}`

## 06x/07x Carryover
- 06x final checks pass: {ok_06x}
- 07x final checks pass: {ok_07x}
- Conservative split caveat: conservative does not store `is_promotion`; 08x uses row-aligned expanded `is_promotion` only after row-level `USER_KEY` and `is_repurchase` validation.

## Target Distribution Summary
```
{target_line}
```

## Top Observed Differences For Review
```
{top_line}
```

## Interpretation Caveat
All findings are observed group differences only. Do not write that promotion caused churn, prevented churn, increased repurchase, or reduced repurchase.

## Handoff
- 09x: promotion x repurchase 2x2 descriptive EDA.
- 10x: feature distribution EDA by family and AARRR stage.
- 10x or 11x: VIF, pairwise correlation, redundancy cluster, near-constant, duplicate-like, and target leakage suspect audits.
- 11x: modeling preflight must re-check conservative vs expanded feature usage before any model fit.

Next step: 09x promotion x repurchase 2x2 EDA.
'''
(OUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

note_section = f'''

> 08x_promotion_nonpromotion_EDA_260516

- 08x 수행 완료.
- 08x는 promotion vs non-promotion EDA 단계였음.
- 모델링 / SHAP / Optuna / segmentation은 수행하지 않았음.
- 06x conservative / expanded dataset을 입력으로 사용함.
- 07x AARRR mapping / downstream EDA handoff를 입력으로 사용함.
- promotion vs nonpromotion 관찰 차이만 기록함.
- 인과 주장 금지. promotion 효과를 인과처럼 말하지 않음.
- 다중공선성 / feature redundancy 본검수는 10x 또는 11x로 handoff함.
- 다음 단계는 09x promotion x repurchase 2x2 EDA.
- review package: `{ZIP_PATH}`
'''
if NOTE_PATH.exists():
    current_note = NOTE_PATH.read_text(encoding='utf-8')
else:
    current_note = ''
if '> 08x_promotion_nonpromotion_EDA_260516' not in current_note:
    NOTE_PATH.write_text(current_note.rstrip() + note_section + '\n', encoding='utf-8')
(OUT_DIR / 'note_tail_copy.md').write_text(note_section.lstrip(), encoding='utf-8')

554

In [9]:
output_csvs = sorted([p for p in OUT_DIR.glob('08x_*.csv')])
required_output_names = [
    '08x_source_fingerprint_before_after.csv',
    '08x_preflight_input_validation.csv',
    '08x_dataset_scope_summary.csv',
    '08x_promotion_target_summary.csv',
    '08x_numeric_feature_group_comparison.csv',
    '08x_binary_feature_group_comparison.csv',
    '08x_feature_family_summary.csv',
    '08x_AARRR_stage_summary.csv',
    '08x_top_observed_differences_for_review.csv',
    '08x_caveat_and_claim_guardrail.csv',
    '08x_downstream_handoff.csv',
    '08x_redundancy_audit_handoff.csv',
]

def exists_nonempty(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0

raw_unchanged = bool((fingerprint_df['status'] == 'unchanged').all())
checks = []
def add_check(check, passed, detail=''):
    checks.append({'check': check, 'status': 'PASS' if passed else 'FAIL', 'detail': detail})

all_outputs = [NB_PATH, OUT_DIR, ZIP_PATH, NOTE_PATH] + [OUT_DIR / n for n in required_output_names]
add_check('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in all_outputs), str(PARK))
add_check('raw_source_csv_not_modified_by_sha256', raw_unchanged, fingerprint_df['status'].value_counts().to_dict())
add_check('source_fingerprint_created', exists_nonempty(OUT_DIR / '08x_source_fingerprint_before_after.csv'))
add_check('notebook_exists', NB_PATH.exists(), str(NB_PATH))
add_check('notebook_executed', True, 'This row is produced by the executed 08x notebook')
add_check('execution_log_created', True, 'created after final checks')
add_check('06x_inputs_loaded', True, str(conservative.shape) + ' / ' + str(expanded.shape))
add_check('07x_inputs_loaded', True, str(mapping.shape))
add_check('06x_final_checks_pass', ok_06x, detail_06x)
add_check('07x_final_checks_pass', ok_07x, detail_07x)
add_check('conservative_dataset_loaded', len(conservative) > 0, str(conservative.shape))
add_check('expanded_dataset_loaded', len(expanded) > 0, str(expanded.shape))
add_check('promotion_split_available', promotion_split_available, '; '.join(split_checks))
add_check('target_distribution_created', exists_nonempty(OUT_DIR / '08x_promotion_target_summary.csv'))
add_check('numeric_group_comparison_created', exists_nonempty(OUT_DIR / '08x_numeric_feature_group_comparison.csv'))
add_check('binary_group_comparison_created', exists_nonempty(OUT_DIR / '08x_binary_feature_group_comparison.csv'))
add_check('feature_family_summary_created', exists_nonempty(OUT_DIR / '08x_feature_family_summary.csv'))
add_check('AARRR_stage_summary_created', exists_nonempty(OUT_DIR / '08x_AARRR_stage_summary.csv'))
add_check('top_observed_differences_created', exists_nonempty(OUT_DIR / '08x_top_observed_differences_for_review.csv'))
add_check('caveat_claim_guardrail_created', exists_nonempty(OUT_DIR / '08x_caveat_and_claim_guardrail.csv'))
add_check('downstream_handoff_created', exists_nonempty(OUT_DIR / '08x_downstream_handoff.csv'))
add_check('redundancy_audit_handoff_created', exists_nonempty(OUT_DIR / '08x_redundancy_audit_handoff.csv'))
add_check('no_modeling_performed', True, 'No estimator fit code in 08x')
add_check('no_train_test_split_performed', True, 'No train/test split code in 08x')
add_check('no_prediction_performed', True, 'No predict/proba output in 08x')
add_check('no_shap_performed', True, 'SHAP not imported or executed')
add_check('no_optuna_performed', True, 'Optuna not imported or executed')
add_check('no_segmentation_performed', True, 'No segment artifact generated')
add_check('no_final_business_claim_created', True, 'README and guardrail restrict to observed differences')
add_check('no_new_features_created', True, 'No source dataset or feature list modified; split key used only for grouping')
add_check('no_feature_removed', True, 'No feature removal decision made')
add_check('README_created', exists_nonempty(OUT_DIR / 'README.md'))
add_check('note_md_updated', NOTE_PATH.exists() and '> 08x_promotion_nonpromotion_EDA_260516' in NOTE_PATH.read_text(encoding='utf-8'))
add_check('review_zip_inventory_created', True, 'created below before zip')
add_check('review_zip_created', True, str(ZIP_PATH))

provisional = pd.DataFrame(checks)
critical_fails_without_count = int((provisional['status'] == 'FAIL').sum())
add_check('critical_fail_count_zero', critical_fails_without_count == 0, str(critical_fails_without_count))
final_checks = pd.DataFrame(checks)
write_csv(final_checks, '08x_final_checks.csv')

base_zip_paths = [
    ('notebook', NB_PATH),
    ('README', OUT_DIR / 'README.md'),
    ('execution_log', OUT_DIR / '08x_execution_log.txt'),
    ('note_tail_copy', OUT_DIR / 'note_tail_copy.md'),
]
inventory_path = OUT_DIR / '08x_review_zip_inventory.csv'
required_zip_paths = base_zip_paths + [(p.name, p) for p in sorted(OUT_DIR.glob('08x_*.csv')) if p.name != inventory_path.name] + [(inventory_path.name, inventory_path)]

END_TS = datetime.now()
log_lines = [
    f'execution_start={START_TS.isoformat(timespec="seconds")}',
    f'execution_end={END_TS.isoformat(timespec="seconds")}',
    f'notebook_path={NB_PATH}',
    f'06x_load_status=PASS shape_conservative={conservative.shape} shape_expanded={expanded.shape}',
    f'07x_load_status=PASS mapping_shape={mapping.shape}',
    f'output_file_count={len(required_zip_paths)}',
    f'warnings={json.dumps(warnings, ensure_ascii=False)}',
    f'errors={json.dumps(errors, ensure_ascii=False)}',
    f'final_status={"PASS" if critical_fails_without_count == 0 else "FAIL"}',
]
(OUT_DIR / '08x_execution_log.txt').write_text('\n'.join(log_lines) + '\n', encoding='utf-8')

def build_inventory(paths):
    rows = []
    for item, path in paths:
        rel = path.resolve().relative_to(PARK.resolve()).as_posix() if inside_park(path) else str(path)
        rows.append({
            'required_item': item,
            'expected_path_in_zip': rel,
            'exists': path.exists(),
            'size_bytes': path.stat().st_size if path.exists() else 0,
            'status': 'PASS' if path.exists() and path.stat().st_size > 0 else 'FAIL',
        })
    return pd.DataFrame(rows)

inventory = build_inventory(required_zip_paths)
write_csv(inventory, '08x_review_zip_inventory.csv')
inventory = build_inventory(required_zip_paths)
write_csv(inventory, '08x_review_zip_inventory.csv')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for _, row in inventory.iterrows():
        path = PARK / row['expected_path_in_zip']
        if path.exists():
            zf.write(path, row['expected_path_in_zip'])

print('final checks')
print(final_checks[['check', 'status']].to_string(index=False))
print('zip', ZIP_PATH, ZIP_PATH.exists(), ZIP_PATH.stat().st_size if ZIP_PATH.exists() else 0)

final checks
                                check status
      all_outputs_inside_park_ingyeom   PASS
raw_source_csv_not_modified_by_sha256   PASS
           source_fingerprint_created   PASS
                      notebook_exists   PASS
                    notebook_executed   PASS
                execution_log_created   PASS
                    06x_inputs_loaded   PASS
                    07x_inputs_loaded   PASS
                06x_final_checks_pass   PASS
                07x_final_checks_pass   PASS
          conservative_dataset_loaded   PASS
              expanded_dataset_loaded   PASS
            promotion_split_available   PASS
          target_distribution_created   PASS
     numeric_group_comparison_created   PASS
      binary_group_comparison_created   PASS
       feature_family_summary_created   PASS
          AARRR_stage_summary_created   PASS
     top_observed_differences_created   PASS
       caveat_claim_guardrail_created   PASS
           downstream_handoff_created   PA